# GPU training environment

Welcome. This session has an **emulated NVIDIA L4**. There is no physical GPU anywhere in this environment.

That is worth being clear about up front, because almost everything behaves as though the GPU were real:

| Works properly | Does not, and cannot |
|---|---|
| `nvidia-smi`, `nvtop` | Any real speedup |
| `sbatch`, `squeue`, `scancel`, `sinfo` | Custom CUDA C++ extensions |
| `torch.cuda.*`, `.to("cuda")`, `.cuda()` | `torch.compile` with a CUDA backend, Triton |
| Device memory limits, and real out-of-memory errors | Any timing comparison between GPU and CPU |
| `@cuda.jit` kernels, via Numba's simulator | Tensor cores, mixed precision speedups |

The arithmetic all runs on the CPU. **Never treat a timing from this environment as a GPU result.**

What this environment is *for* is the part of GPU work that does not depend on the hardware being present: asking a scheduler for a device, checking you actually got one, reading utilisation and memory, recognising an out-of-memory error, and writing correct kernel code.

## 1. Is there a GPU?

The first thing to run in any GPU session.

In [ ]:
!nvidia-smi

Read that output carefully, because you will read it often:

- **top right** — the driver and CUDA version. Your software has to be built against something compatible.
- **`Memory-Usage`** — how much of the card's 23 GB is in use. This is the number that decides your batch size.
- **`GPU-Util`** — the fraction of recent time the device was executing anything at all. A low number during training usually means the GPU is waiting on data, not that the work is small.
- **`Processes`** — who is using the card. On a shared node, this tells you whether you are alone on it.

Note `Fan` reads `N/A`: the L4 is a passive card cooled by server airflow, so it has no fan of its own to report on. That is genuine, not a gap in the emulator.

## 2. A live view

`nvidia-smi` is a snapshot. `nvtop` is the live version, and it is much better for watching what a job actually does.

Open a terminal (**File → New → Terminal**) and run:

```bash
nvtop
```

Leave it running in one window while you work in another. Press `q` to quit.

To give it something to show, run this in a *second* terminal and watch the graphs move:

```bash
gpuemu-burn --time 60 --memory 4GiB
```

## 3. The GPU from Python

In [ ]:
import gpuemu.torch_shim  # noqa: F401  - before torch, sets up the CUDA API
import torch

print(f"CUDA available : {torch.cuda.is_available()}")
print(f"Device count   : {torch.cuda.device_count()}")
print(f"Device name    : {torch.cuda.get_device_name(0)}")
print(f"Capability     : {torch.cuda.get_device_capability(0)}")

props = torch.cuda.get_device_properties(0)
print(f"Total memory   : {props.total_memory / 1024**3:.1f} GiB")
print(f"SMs            : {props.multi_processor_count}")

Now put something on the device and watch the memory figure move. Keep `nvtop` open in a terminal while you run the next cell.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

before = torch.cuda.memory_allocated()

# 4096 x 4096 float32 = 64 MiB
x = torch.randn(4096, 4096, device=device)
y = torch.randn(4096, 4096, device=device)

after = torch.cuda.memory_allocated()
print(f"Allocated by those two tensors: {(after - before) / 1024**2:.0f} MiB")
print(f"Total on device now          : {after / 1024**2:.0f} MiB")

In [ ]:
!nvidia-smi --query-gpu=memory.used,memory.total,utilization.gpu --format=csv

That `--query-gpu` form is the one to remember. It is how you log GPU usage from inside a job script, and it is far easier to parse than the table.

## 4. Running out of memory

Device memory is the constraint you will hit most often, and the error is one you should recognise on sight. The limit here is enforced for real, so this cell fails the way it would on hardware.

In [ ]:
try:
    # ~40 GiB on a 23 GiB card
    hopeless = torch.zeros(100_000, 100_000, device=device)
except RuntimeError as exc:
    print(f"{type(exc).__name__}:\n{exc}")

When you hit that for real, the options are roughly, in order of what to try first:

1. **Reduce the batch size.** Almost always the fastest fix.
2. **Free what you are done with** — `del tensor`, then `torch.cuda.empty_cache()`.
3. **Use gradient accumulation** to keep the effective batch size while shrinking the real one.
4. **Use mixed precision** to halve activation memory.
5. **Ask for a bigger card**, if your site has one.

Free the tensors and confirm the memory comes back:

In [ ]:
del x, y
torch.cuda.empty_cache()
print(f"Allocated now: {torch.cuda.memory_allocated() / 1024**2:.0f} MiB")

## 5. Where next

- **`01-submitting-jobs.ipynb`** — `sbatch`, `squeue`, and the one mistake everybody makes.
- **`02-cuda-kernels.ipynb`** — writing real `@cuda.jit` kernels that really execute.
- **`~/gpu-training/examples/`** — job scripts to submit and pick apart.

One last time, because it matters: the device is emulated, and nothing here tells you how fast anything would run on a real L4.